### Arboles 
A grandes rasgos los árboles de clasificación son métodos que particionan el espacio de covariables $\mathcal{X}$ en piezas disjuntas y entonces clasificamos las observaciones de acuerdo a la partición a la que pertencen.

Supongamos que se tinen dos categorias  $y\in \mathcal{Y}=\{0,1\}$. Y tenemos solamente una covariable $X$. Escogemos un punto t que divide a la recta real en dos conjunto $A_1=(-\infty,t]$, $A_2=(t,\infty)$. Sea $\hat{p}_s(j)$ la proporción de observaciones en $A_s$, tal que $Y_i=j$. 
$$
\hat{p}_s(j)=\dfrac{\sum_{i=1}^n I(Y_i=j,X_i\in A_s)}{\sum_{i=1}^n I(X_i\in A_s)}
$$
para $s=1,2$ y $j=0,1$.

Ejemplo: Proporción de observaciones en $A_1$ tal que Y=0.
$$
\hat{p}_1(j=0)=\dfrac{\sum_{i=1}^n I(Y_i=0,X_i\in A_1)}{\sum_{i=1}^n I(X_i\in A_1)}
$$

La impureza del split se define como, 
$$
I(t)=\sum_{s=1}^2\gamma_s
$$
donde $\gamma_s=1-\sum_{j=0}^1 \hat{p}_s(j)^2$.
Esta es una medida de la impureza y se le conoce como indice de $\textbf{Gini}$. Si una partición $A_s$ contiene todos los 0's o 1's entonces $\gamma_s=0$. De otra manera $\gamma_s>0$ Entonces nosotros tratamos de escoger un punto $t$ tal que minimize la impureza.



Ventajas

* Fácil de interpretar.
* No paramétrico

Deventajas 

* Sobreajuste
* Pérdida de información al categorizar variables continuas
* Precisión: si pueden usar mejor SVM, usenlos! en ocaciones tienen tasas de error más bajas hasta 30 %  
* Inestabilidad: un pequeño cambio en los datos puede modificar ampliamente la estructura del árbol
* hay que tener cuidado con la heurística 

In [0]:
import os 
#agregar al path el lugar de la instalacion de graphviz (dot.exe)
os.environ["PATH"] += os.pathsep + r'C:\Program Files\Graphviz\bin'

In [0]:
#!pip install graphviz

In [0]:
import urllib.request          as     url
import pandas                  as     pd
from   sklearn                 import tree
import numpy                   as     np
from   sklearn.metrics         import (
        accuracy_score,
        recall_score,
        precision_recall_curve,
        average_precision_score,
        f1_score,
        precision_score,
        confusion_matrix,
        ConfusionMatrixDisplay,)

from   sklearn.model_selection import train_test_split
from   sklearn.ensemble        import BaggingClassifier,RandomForestClassifier
import matplotlib.pylab        as     plt 
from   sklearn.model_selection import GridSearchCV,RandomizedSearchCV,StratifiedKFold
import sklearn
from scipy.stats import randint
from sklearn.tree import plot_tree


#### Descripción de los datos
Los datos del Estudio de factores de riesgo coronario (CORIS) incluyen a 462 hombres de entre 15 y 64 años de tres zonas rurales de Sudáfrica (Rousseauw et al. (1983)). La variable de respuesta Y es la presencia (Y = 1) o ausencia (Y = 0) de enfermedad coronaria. Hay 9 covariables:

* sbp- presión arterial sistólica
* tobacco tabaquismo acumulado (kg)
* ldl (colesterol unido a lipoproteínas de baja densidad)
* adiposity, adiposidad
* famhist (antecedentes familiares de enfermedad cardíaca)
* tipoa (comportamiento tipo A), 
* obesity, obesidad 
* alcohol (consumo actual de alcohol) 
* age

In [0]:
data=pd.read_csv('coris.csv',sep=',',header=0)
data.shape
data.head(5)

Los hiperparámetros son los parámetros de configuración externos a un modelo de aprendizaje, cuyos valores no se estiman a partir de los datos durante el proceso de entrenamiento, sino que deben ser definidos por el usuario antes de iniciar dicho proceso
Parametros
1. max_leaf_nodes es el maximo número de nodos hojas (nodos sin hijos) para reducción relativa de la impureza
2. max_depth profundidad maxima.
3. criterion criterio para la división de los datos, Gini (impureza)
El indice de Gini es un valor entre 0 y 1

In [0]:
y=data.pop("chd")
X=data.drop("row.names",axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [0]:
print('Personas con enferdad de las coronarias')
print(pd.value_counts(y_train))
print('Personas con edad menor de 50 años')
print(sum([1 for i in X_train['age'] if i<50]))

In [0]:
#ccp_alpha= 0.015 agregar para mejorar la clasificacion max_leaf_nodes=10
tr=tree.DecisionTreeClassifier(criterion="gini",max_depth=5)
trf=tr.fit(X_train,y_train)

In [0]:
plt.figure(figsize=(20, 8))

plot_tree(
    trf,
    feature_names=X_train.columns,
    class_names=["ausencia", "presencia"],
    filled=True,
    rounded=True,
    max_depth=5,
    fontsize=10
)

plt.show()


### Entropía de Shannon

La entropía fue introducida por Claude Shannon en 1948 en su trabajo A Mathematical Theory of Communication.  
Se define como una medida del grado de incertidumbre o información promedio contenida en una variable aleatoria.

Para una variable discreta \( Y \) con posibles valores $$ y_1, y_2, \ldots, y_K $$ y probabilidades $$ p(y_i) $$:

$$
H(Y) = - \sum_{i=1}^{K} p(y_i) \log_2 p(y_i)
$$



Cada término $$ -\log_2 p(y_i) $$ mide la cantidad de información (en bits) asociada a observar el evento $$ y_i $$.

- Si un evento es muy probable (por ejemplo \( p = 0.9 \)), su ocurrencia no aporta mucha información nueva:  
$$  -\log_2(0.9) \approx 0.15 \text{ bits}$$
- Si un evento es muy improbable (por ejemplo \( p = 0.01 \)), su ocurrencia aporta mucha información:  
$$
  -\log_2(0.01) = 6.64 \text{ bits}
$$

En otras palabras, la información es inversamente proporcional a la probabilidad:  
cuanto menos probable es algo, más información obtenemos al observarlo.

---

La entropía \( H(Y) \) es entonces el promedio ponderado de esa sorpresa:

$$
H(Y) = \mathbb{E}[-\log_2 p(Y)]
$$

Por eso se interpreta como la cantidad promedio de bits necesarios para codificar el valor de Y.


In [0]:
yp=trf.predict(X_test)
print('Porcentaje de datos bien clasificados')
print(accuracy_score(y_test,yp))
print('número de datos bien clasificados')
recall_score(y_test,yp)

In [0]:
1 0 0 0 1  0 0 | 1 0 0 1 1 |  1 0 1 1 |1 0 1 0 0
              0.45      0.6           0.8

###  Exactitud

La exactitud o accuracy es la proporción de todas las clasificaciones correctas, ya sean positivas o negativas. 
Se define matemáticamente como:

$$
\text{Exactitud} = \frac{\text{clasificaciones correctas}}{\text{clasificaciones totales}} 
= \frac{TP + TN}{TP + TN + FP + FN}
$$


Un modelo perfecto tendría cero falsos positivos y cero falsos negativos, y por lo tanto una exactitud de 1.0 o 100%.
Dado que incorpora los cuatro resultados de la matriz de confusión ($TP, FP, TN, FN$), y dado un conjunto de datos equilibrado, con un número similar de ejemplos en ambas clases, la precisión puede servir como una medida general de la calidad del modelo.
Por esta razón, suele ser la métrica de evaluación predeterminada para modelos genéricos o no especificados que realizan tareas de clasificación generales.
Sin embargo, cuando el conjunto de datos está desequilibrado, o cuando un tipo de error (FN o FP) es más costoso que el otro,  
como sucede en la mayoría de las aplicaciones reales, es mejor optimizar otras métricas (por ejemplo, recall o F1-score).


Para conjuntos de datos muy desbalanceados, donde una clase aparece muy raramente (por ejemplo, el 1% del tiempo),  
un modelo que predice resultados negativos el 100% del tiempo obtendrá una precisión del 99%, a pesar de ser totalmente inútil.

---

### Recuperación o sensibilidad, o tasa de positivos verdaderos

La tasa de positivos verdaderos (TPR), o la proporción de todos los positivos reales que se clasificaron correctamente como positivos, también se conoce como recuperación.


El recuerdo se define matemáticamente como:

$$
\text{Recall (o TPR)} = \frac{\text{positivos reales clasificados correctamente}}{\text{total de positivos reales}} = \frac{TP}{TP + FN}
$$

Los falsos negativos son positivos reales que se clasificaron erróneamente como negativos, por lo que aparecen en el denominador.  
La recuperación también se denomina probabilidad de detección: responde a la pregunta  
"¿Qué fracción de verdaderos positivos detecta este modelo?".

Un modelo hipotético perfecto tendría cero falsos negativos y por tanto un recall (TPR) de 1.0, es decir, una tasa de detección del 100%.

En un conjunto de datos desequilibrado donde el número de positivos reales es muy bajo, la recuperación es una métrica más significativa que la precisión, ya que mide la capacidad del modelo para identificar correctamente todos los casos positivos.  
Para aplicaciones como la predicción de enfermedades, la correcta identificación de los casos positivos es crucial.  
Un falso negativo suele tener consecuencias más graves que un falso positivo.  

---

### Tasa de falsos positivos

La tasa de falsos positivos (TPP) es la proporción de todos los negativos reales que se clasificaron incorrectamente como positivos, también conocida como probabilidad de falsa alarma.  
Se define matemáticamente como:

$$
\text{FPR} = \frac{\text{negativos reales clasificados incorrectamente}}{\text{total de negativos reales}} = \frac{FP}{FP + TN}
$$

Los falsos positivos son negativos que se clasificaron incorrectamente, por lo que aparecen en el denominador. 
Un modelo perfecto tendría cero falsos positivos y por tanto un FPR de 0.0, es decir, una tasa de falsas alarmas del 0%.
En un conjunto de datos desequilibrado donde el número de negativos reales es muy, muy bajo (digamos 1 o 2 ejemplos en total),  
el FPR es menos significativo y menos útil como métrica.

### Precisión

La precisión es la proporción de todas las clasificaciones positivas del modelo que son realmente positivas.  
Se define matemáticamente como:

$$
\text{Precisión} = \frac{\text{positivos reales clasificados correctamente}}{\text{total de clasificados como positivos}} = \frac{TP}{TP + FP}
$$

Un modelo hipotético perfecto tendría cero falsos positivos y, por lo tanto, una precisión de 1.0.
En un conjunto de datos desequilibrado donde el número de resultados positivos reales es muy, muy bajo (digamos 1 o 2 ejemplos en total), la precisión es menos significativa y menos útil como métrica.

La precisión mejora a medida que disminuyen los falsos positivos, mientras que la recuperación mejora cuando disminuyen los falsos negativos.  
Sin embargo, como se vio en la sección anterior, aumentar el umbral de clasificación tiende a disminuir el número de falsos positivos y a aumentar el de falsos negativos, mientras que disminuir el umbral tiene el efecto contrario. Como resultado, la precisión y la recuperación suelen mostrar una relación inversa: mejorar una empeora la otra.

### F1-Score

El F1-Score es una métrica que combina la precisión y la recuperación en una sola medida.  
Se define como la media armónica entre la precisión (Precision) y la recuperación (Recall).

Matemáticamente, se expresa como:

$$
F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}
$$

El F1-Score toma valores entre 0 y 1.

- Un valor de 1 indica una perfecta precisión y recuperación.
- Un valor cercano a 0 indica un desempeño deficiente en al menos una de las dos métricas.


Esta métrica es especialmente útil cuando existe un desequilibrio entre las clases,  
ya que penaliza los modelos que logran una alta precisión pero una baja recuperación, o viceversa.
En resumen, el F1-Score busca un equilibrio entre la capacidad del modelo para identificar correctamente los casos positivos (recall) y la capacidad para evitar clasificar incorrectamente casos negativos como positivos (precision).




In [0]:
cm = confusion_matrix(y_test, yp,normalize='true')
print("\nMatriz de confusión:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Matriz de confusión")
plt.show()

print("\nOtras métricas:")
print(f"Precisión: {precision_score(y_test, yp):.3f}")
print(f"Sensibilidad o Recall: {recall_score(y_test, yp):.3f}")
print(f"F1-Score: {f1_score(y_test, yp):.3f}")

### Hiperparámetros

Los hiperparámetros son parámetros externos al modelo que controlan su proceso de aprendizaje y su estructura interna, pero no se estiman directamente a partir de los datos.  
A diferencia de los parámetros del modelo como los coeficientes en una regresión lineal, los hiperparámetros deben ser fijados antes del entrenamiento.
Los hiperparámetros son **variables de alto nivel** que definen el comportamiento del modelo y del algoritmo de optimización.  
Ejemplos comunes incluyen:

- La profundidad máxima (`max_depth`) en un árbol de decisión,  
- La tasa de aprendizaje (`learning_rate`) en modelos de boosting o redes neuronales,  
- El parámetro de regularización (`C`, `alpha`, `lambda`) en modelos lineales o de margen máximo.

Su ajuste correcto es fundamental para lograr un equilibrio entre **sesgo y varianza**, y evitar tanto el **sobreajuste** como el **subajuste**.


La búsqueda GridSearchCV funciona como un rastreo exhaustivo que prueba todas las combinaciones posibles de una lista predefinida de valores para cada hiperparámetro. Formalmente, este proceso genera un producto cartesiano que cubre cada intersección de una cuadrícula de búsqueda, evaluando el desempeño mediante validación cruzada en cada punto.

La búsqueda aleatoria o RandomizedSearchCV selecciona configuraciones al azar basándose en distribuciones de probabilidad en lugar de seguir una estructura rígida. En cada iteración, el algoritmo toma valores aleatorios para los hiperparámetros, lo que permite explorar una variedad mucho más amplia de configuraciones en menos tiempo.

In [0]:
param_grid = {
    'max_depth' : [5,6,7,8,9,10,11,12,13,14],
    'max_leaf_nodes' : [i for i in range(7,20,1)],
    'criterion':["gini", "entropy"]
}
model = GridSearchCV(tree.DecisionTreeClassifier(),param_grid, scoring = 'recall',n_jobs=-1 )
model.fit(X_train, y_train)

In [0]:
print("model score: %.3f" % model.score(X_test, y_test))
print ("hiperparametros: ",str(model.best_params_))

In [0]:
param_dist = {
    'max_depth': [5,6,7,8,9,10,11,12,13,14],           
    'max_leaf_nodes': [i for i in range(7,20,1)],     
    'criterion': ["gini", "entropy"]      
}

dt = tree.DecisionTreeClassifier()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


random_search = RandomizedSearchCV(
    estimator=dt,
    param_distributions=param_dist,
    n_iter=150,                  
    scoring='recall',          
    cv=cv,     
    random_state=42,   
    n_jobs=-1                   
)


random_search.fit(X_train, y_train)

print("Mejor score:", random_search.best_score_)
print("Mejores hiperparámetros encontrados:", random_search.best_params_)


In [0]:
best_model = random_search.best_estimator_
y_prob = best_model.predict_proba(X_test)[:, 1]   

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(recall, precision, color='blue', label=f'Curva Precision–Recall (AP = {ap:.2f})')
plt.fill_between(recall, precision, alpha=0.2, color='blue')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision–Recall')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

### Poda de costo mínimo y complejidad

La **poda de costo mínimo y complejidad** es un algoritmo que se utiliza para **podar un árbol de decisión** y evitar el sobreajuste (*overfitting*).  
Este algoritmo está parametrizado por \( \alpha \geq 0 \), conocido como **parámetro de complejidad**.  

El parámetro de complejidad se utiliza para definir la **medida de costo-complejidad**, \( R_\alpha(T) \), de un árbol \( T \), dada por:

$$
R_\alpha(T) = R(T) + \alpha |T|
$$

donde \( |T| \) es el número de nodos terminales en \( T \), y \( R(T) \) se define tradicionalmente como la **tasa total de clasificación errónea** en los nodos terminales.

---

El objeto `DecisionTreeClassifier` de **scikit-learn** proporciona parámetros como  
`min_samples_leaf` y `max_depth` para evitar que un árbol se sobreajuste.  

La **poda de costo-complejidad** ofrece otra opción para controlar el tamaño del árbol.  
En `DecisionTreeClassifier`, esta técnica de poda está controlada por el parámetro de complejidad de costo \( \text{ccp\_alpha} \).

---


In [0]:
path = tr.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

In [0]:
fig, ax = plt.subplots()
ax.plot(ccp_alphas[:-1], impurities[:-1], marker='o', drawstyle="steps-post")
ax.set_xlabel("effective alpha")
ax.set_ylabel("total de impureza")
ax.set_title("Impureza vs effective alpha")

In [0]:
models = []
for ccp_alpha in ccp_alphas:
    mod = tree.DecisionTreeClassifier(random_state=0, ccp_alpha=ccp_alpha)
    mod.fit(X_train, y_train)
    models.append(mod)
print("Numero de nodos en el ultimo arbol es: {} con ccp_alpha: {}".format(
      models[-1].tree_.node_count, ccp_alphas[-1]))

In [0]:
models = models[:-1]
ccp_alphas = ccp_alphas[:-1]

In [0]:
node_counts = [mod.tree_.node_count for mod in models]
depth = [mod.tree_.max_depth for mod in models]
fig, ax = plt.subplots(2, 1)
ax[0].plot(ccp_alphas, node_counts, marker='o', drawstyle="steps-post")
ax[0].set_xlabel("alpha")
ax[0].set_ylabel("Numero de nodos")
ax[0].set_title("Numero de nodos vs alpha")
ax[1].plot(ccp_alphas, depth, marker='o', drawstyle="steps-post")
ax[1].set_xlabel("alpha")
ax[1].set_ylabel("profundidad del arbol")
ax[1].set_title("profundidad vs alpha")
fig.tight_layout()

In [0]:
train_scores = [mod.score(X_train, y_train) for mod in models]
test_scores = [mod.score(X_test, y_test) for mod in models]

fig, ax = plt.subplots()
ax.set_xlabel("alpha")
ax.set_ylabel("accuracy")
ax.set_title("Accuracy vs alpha para datos de training y test")
ax.plot(ccp_alphas, train_scores, marker='o', label="train",
        drawstyle="steps-post")
ax.plot(ccp_alphas, test_scores, marker='o', label="test",
        drawstyle="steps-post")
ax.legend()
plt.show()

In [0]:
train_recalls = [recall_score(y_train, mod.predict(X_train)) for mod in models]
test_recalls = [recall_score(y_test, mod.predict(X_test)) for mod in models]

fig, ax = plt.subplots()
ax.set_xlabel("alpha")
ax.set_ylabel("recall")
ax.set_title("recall vs alpha para datos de training y test")
ax.plot(ccp_alphas, train_recalls, marker='o', label="train",
        drawstyle="steps-post")
ax.plot(ccp_alphas, test_recalls, marker='o', label="test",
        drawstyle="steps-post")
ax.legend()
plt.show()

In [0]:
#ccp_alpha= 0.015 agregar para mejorar la clasificacion max_leaf_nodes=10
tr=tree.DecisionTreeClassifier(criterion="entropy",max_depth=11,ccp_alpha=0.013,max_leaf_nodes= 8)
trf=tr.fit(X_train,y_train)
y_proba = trf.predict_proba(X_test)[:, 1]

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)


indice_optimo = np.argmax(f1_scores[:-1])
umbral_optimo = thresholds[indice_optimo]

print("calibracion")
print(f"Umbral optimo encontrado: {umbral_optimo:.4f}")
print(f"F1-Score: {f1_scores[indice_optimo]:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision[:-1], label="Precisión", color='blue')
plt.plot(thresholds, recall[:-1], label="Recall", color='orange')
plt.plot(thresholds, f1_scores[:-1], label="F1-Score", color='green', linestyle='--')
plt.axvline(x=umbral_optimo, color='red', linestyle=':', label=f'Umbral Óptimo ({umbral_optimo:.2f})')

plt.title("Precisión, recall y f1-score vs0 umbral de decisión")
plt.xlabel("Umbral de decisión")
plt.ylabel("Puntuación")
plt.legend(loc="best")
plt.grid(True)
plt.show()

Ojo: Scikit-Learn usa una versión optimizada del algoritmo CART el cual es un algoritmo muy similar a C4.5, sin embargo esta implementación no soporta variables categoricas por ahora.